# Recomendacion de productos con similitud del coseno

Esta libreta usa el dataset `similitud_de_coseno_Productos.csv` para recomendar productos parecidos.

Flujo:

1. Cargar el dataset.
2. Limpiar valores vacios o `NULL`.
3. Crear un texto ponderado por producto.
4. Convertir el texto a vectores con TF-IDF.
5. Calcular similitud del coseno.
6. Consultar recomendaciones por producto o por texto de busqueda.


## 1. Importar librerias

In [ ]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Cargar el dataset

In [ ]:
DATASET_PATH = Path("similitud_de_coseno_Productos.csv")

if not DATASET_PATH.exists():
    DATASET_PATH = Path("docs/similitud_de_coseno_Productos.csv")

df = pd.read_csv(DATASET_PATH, na_values=["NULL", "null", ""])

print(f"Dataset cargado: {DATASET_PATH}")
print(f"Filas: {len(df)}")
df.head()


## 3. Validar columnas y limpiar datos

In [ ]:
COLUMNAS_REQUERIDAS = [
    "id_producto",
    "producto",
    "productType",
    "categoryName",
    "brandName",
    "price",
    "description",
    "features",
    "supplementFlavor",
    "supplementPresentation",
    "supplementServings",
    "apparelSize",
    "apparelColor",
    "apparelMaterial",
    "texto_recomendacion",
]

faltantes = [col for col in COLUMNAS_REQUERIDAS if col not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas: {faltantes}")

df = df.copy()
df["id_producto"] = df["id_producto"].astype(int)
df["price"] = pd.to_numeric(df["price"], errors="coerce").fillna(0)

columnas_texto = [col for col in COLUMNAS_REQUERIDAS if col not in ["id_producto", "price"]]
df[columnas_texto] = df[columnas_texto].fillna("")

resumen = {
    "filas": len(df),
    "ids_duplicados": int(df["id_producto"].duplicated().sum()),
    "texto_recomendacion_vacio": int((df["texto_recomendacion"].str.strip() == "").sum()),
    "productos_vacios": int((df["producto"].str.strip() == "").sum()),
}

resumen


## 4. Crear texto para el modelo

Se ponderan algunos campos repitiendolos. Esto ayuda a que tipo, categoria, marca y nombre pesen mas que datos secundarios.

In [ ]:
def unir_texto(*valores):
    return " ".join(str(valor).strip() for valor in valores if str(valor).strip())


def crear_texto_modelo(row):
    return unir_texto(
        row["producto"], row["producto"], row["producto"],
        row["productType"], row["productType"], row["productType"],
        row["categoryName"], row["categoryName"], row["categoryName"],
        row["brandName"], row["brandName"],
        row["description"],
        row["features"],
        row["supplementFlavor"],
        row["supplementPresentation"],
        row["supplementServings"],
        row["apparelSize"],
        row["apparelColor"],
        row["apparelMaterial"],
        row["texto_recomendacion"],
    )


df["texto_modelo"] = df.apply(crear_texto_modelo, axis=1)
df[["id_producto", "producto", "texto_modelo"]].head()


## 5. Vectorizar con TF-IDF

In [ ]:
STOPWORDS_ES = [
    "a", "al", "algo", "ante", "con", "como", "de", "del", "desde", "durante",
    "e", "el", "ella", "en", "entre", "es", "esta", "este", "esto", "ideal",
    "la", "las", "lo", "los", "mas", "mejor", "para", "por", "que", "se", "sin",
    "su", "sus", "un", "una", "unas", "unos", "y",
]

vectorizador = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    stop_words=STOPWORDS_ES,
    ngram_range=(1, 2),
    min_df=1,
)

matriz_tfidf = vectorizador.fit_transform(df["texto_modelo"])

print(f"Matriz TF-IDF: {matriz_tfidf.shape[0]} productos x {matriz_tfidf.shape[1]} terminos")


## 6. Calcular similitud del coseno

In [ ]:
matriz_similitud = cosine_similarity(matriz_tfidf)

indice_por_id = pd.Series(df.index, index=df["id_producto"]).to_dict()

print(f"Matriz de similitud: {matriz_similitud.shape}")


## 7. Funcion para recomendar por id_producto

In [ ]:
COLUMNAS_SALIDA = [
    "id_producto",
    "producto",
    "productType",
    "categoryName",
    "brandName",
    "price",
    "description",
]


def recomendar_por_id(id_producto, top_n=5, mismo_tipo=True):
    if id_producto not in indice_por_id:
        raise ValueError(f"No existe el producto con id_producto={id_producto}")

    idx = indice_por_id[id_producto]
    scores = matriz_similitud[idx]
    producto_base = df.loc[idx]

    candidatos = df.copy()
    candidatos["score_similitud"] = scores
    candidatos = candidatos[candidatos["id_producto"] != id_producto]

    if mismo_tipo:
        candidatos = candidatos[candidatos["productType"] == producto_base["productType"]]

    return (
        candidatos
        .sort_values("score_similitud", ascending=False)
        .head(top_n)[COLUMNAS_SALIDA + ["score_similitud"]]
        .reset_index(drop=True)
    )


## 8. Probar recomendaciones

In [ ]:
ID_PRODUCTO_EJEMPLO = int(df.iloc[0]["id_producto"])

print("Producto base:")
display(df.loc[df["id_producto"] == ID_PRODUCTO_EJEMPLO, COLUMNAS_SALIDA])

print("Recomendaciones:")
recomendar_por_id(ID_PRODUCTO_EJEMPLO, top_n=5, mismo_tipo=True)


## 9. Recomendar por texto libre

Esta funcion sirve para buscar algo como `proteina chocolate`, `creatina sin sabor`, `playera negra` o `short deportivo`.

In [ ]:
def normalizar_texto(valor):
    texto = str(valor).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    return "".join(caracter for caracter in texto if not unicodedata.combining(caracter))


def recomendar_por_busqueda(texto, top_n=5, product_type=None):
    texto = str(texto).strip()
    if not texto:
        raise ValueError("La busqueda no puede estar vacia")

    vector_busqueda = vectorizador.transform([texto])
    scores = cosine_similarity(vector_busqueda, matriz_tfidf).ravel()

    candidatos = df.copy()
    candidatos["score_similitud"] = scores

    if product_type:
        tipo_normalizado = normalizar_texto(product_type)
        candidatos = candidatos[
            candidatos["productType"].apply(normalizar_texto) == tipo_normalizado
        ]

    return (
        candidatos
        .sort_values("score_similitud", ascending=False)
        .head(top_n)[COLUMNAS_SALIDA + ["score_similitud"]]
        .reset_index(drop=True)
    )


In [ ]:
recomendar_por_busqueda("proteina chocolate", top_n=5, product_type="Suplementacion")


## 10. Exportar una prueba de recomendaciones

Opcional: genera un CSV con recomendaciones para un producto.

In [ ]:
salida = recomendar_por_id(ID_PRODUCTO_EJEMPLO, top_n=10, mismo_tipo=True)
salida_path = DATASET_PATH.with_name("recomendaciones_ejemplo.csv")
salida.to_csv(salida_path, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {salida_path.resolve()}")
salida


## Siguiente paso para implementarlo en el sistema

Cuando el resultado de esta libreta sea correcto, el backend puede implementar una ruta parecida a:

`GET /api/products/:id/recommendations`

La logica seria la misma: cargar productos activos, construir el texto, vectorizar, calcular similitud y devolver el top de productos recomendados.